# OOP Challenge Project — Mini Library Management System
**Student:** Nawaraj Tamang  

Combines all four OOP pillars:
- **Abstraction** — `LibraryItem(ABC)` defines a `describe()` contract.
- **Inheritance** — `Book`, `DVD`, `Magazine` all inherit from `LibraryItem`.
- **Polymorphism** — each subclass overrides `describe()` differently.
- **Encapsulation** — `Member` keeps a private `__borrowed_items` list.

## Step 1: The abstract `LibraryItem` base class

In [17]:
from abc import ABC, abstractmethod

class LibraryItem(ABC):
    """Abstract base class for anything the library lends out."""

    def __init__(self, title):
        self.title = title
        self._checked_out = False  # protected: subclasses/Library may read/flip this

    @abstractmethod
    def describe(self):
        pass

    def is_checked_out(self):
        return self._checked_out

## Step 2: Concrete item types (inheritance + polymorphism)

In [18]:
class Book(LibraryItem):
    def __init__(self, title, author):
        super().__init__(title)
        self.author = author

    def describe(self):
        return f"Book: '{self.title}' by {self.author}"


class DVD(LibraryItem):
    def __init__(self, title, runtime_minutes):
        super().__init__(title)
        self.runtime_minutes = runtime_minutes

    def describe(self):
        return f"DVD: '{self.title}' ({self.runtime_minutes} min)"


class Magazine(LibraryItem):
    def __init__(self, title, issue_number):
        super().__init__(title)
        self.issue_number = issue_number

    def describe(self):
        return f"Magazine: '{self.title}' (Issue #{self.issue_number})"

## Step 3: `Member` with a private borrowed-items list (encapsulation)

In [19]:
class Member:
    """A library member with a private list of currently borrowed items."""

    def __init__(self, name):
        self.name = name
        self.__borrowed_items = []  # private: only this class manages it directly

    def borrow(self, item):
        if item.is_checked_out():
            raise ValueError(f"'{item.title}' is already checked out")
        item._checked_out = True
        self.__borrowed_items.append(item)

    def return_item(self, item):
        if item not in self.__borrowed_items:
            print(f"{self.name} did not borrow '{item.title}', nothing to return")
            return
        item._checked_out = False
        self.__borrowed_items.remove(item)

    def get_borrowed_items(self):
        # returns a copy, not the private list itself, so callers can't
        # mutate our internal state directly
        return list(self.__borrowed_items)

## Step 4: `Library` — holds items/members and mediates checkouts

In [20]:
class Library:
    """Holds items and members, and mediates checkouts between them."""

    def __init__(self):
        self.items = []
        self.members = []

    def add_item(self, item):
        self.items.append(item)

    def add_member(self, member):
        self.members.append(member)

    def checkout(self, member, item):
        member.borrow(item)  # raises ValueError if already checked out
        print(f"{member.name} checked out: {item.describe()}")

    @classmethod
    def from_catalog(cls, catalog):
        """Alternative constructor: build a Library from a list of item dicts."""
        library = cls()
        type_map = {"book": Book, "dvd": DVD, "magazine": Magazine}

        for entry in catalog:
            item_type = entry["type"]
            item_class = type_map[item_type]

            if item_type == "book":
                item = item_class(entry["title"], entry["author"])
            elif item_type == "dvd":
                item = item_class(entry["title"], entry["runtime_minutes"])
            else:
                item = item_class(entry["title"], entry["issue_number"])

            library.add_item(item)

        return library

## Step 5: Put it all together

In [21]:
catalog_data = [
    {"type": "book", "title": "Book1", "author": "Author1"},
    {"type": "dvd", "title": "DVD1", "runtime_minutes": 148},
    {"type": "magazine", "title": "Mag1", "issue_number": 245},
]

library = Library.from_catalog(catalog_data)
nawaraj = Member("Nawaraj")
library.add_member(nawaraj)

print("Library catalog:")
for item in library.items:
    # same describe() call, different behavior per subclass - polymorphism
    print(f"  {item.describe()}")

Library catalog:
  Book: 'Book1' by Author1
  DVD: 'DVD1' (148 min)
  Magazine: 'Mag1' (Issue #245)


In [22]:
book_to_borrow = library.items[0]
library.checkout(nawaraj, book_to_borrow)

print(f"{nawaraj.name}'s borrowed items:")
for item in nawaraj.get_borrowed_items():
    print(f"  {item.describe()}")

Nawaraj checked out: Book: 'Book1' by Author1
Nawaraj's borrowed items:
  Book: 'Book1' by Author1


In [25]:
# trying to borrow the same item again should fail cleanly
try:
    library.checkout(Member("Sabina"), book_to_borrow)
except ValueError as error:
    print(f"Checkout blocked as expected: {error}")

Sabina checked out: Book: 'Book1' by Author1


In [24]:
nawaraj.return_item(book_to_borrow)
print(f"After returning, {nawaraj.name} has borrowed: {nawaraj.get_borrowed_items()}")

After returning, Nawaraj has borrowed: []


In [26]:
# confirm the abstract base class can't be instantiated directly
try:
    LibraryItem("Some Title")
except TypeError as error:
    print(f"As expected, this failed: {error}")

As expected, this failed: Can't instantiate abstract class LibraryItem without an implementation for abstract method 'describe'


### What I Learned
This project needed all four pillars working together, not in isolation. The hardest part was deciding up front which class owns which responsibility: the item owns its own `_checked_out` flag, the member owns its private borrowed list, and the library just coordinates between them via `checkout()`. Getting that separation right made the rest of the code fall into place naturally.